In [1]:
from langsmith import evaluate, traceable, wrappers, Client
#from openai import OpenAI
# Assumes you've installed pydantic
from pydantic import BaseModel
from groq import Groq
import os

api_key = os.environ["GROQ_API_KEY"]

client = Groq(
    api_key=api_key)



In [5]:
def valid_reasoning(inputs: dict, outputs: dict) -> bool:
  """Use an LLM to judge if the reasoning and the answer are consistent."""

  instructions = """\

Given the following question, answer, and reasoning, determine if the reasoning \
for the answer is logically valid and consistent with question and the answer.\
"""

  msg = f"Question: {inputs['question']}\nAnswer: {outputs['answer']}\nReasoning: {outputs['reasoning']}"
  
  """response = client.beta.chat.completions.parse(
    model="meta-llama/llama-4-maverick-17b-128e-instruct",
    messages=[{"role": "system", "content": instructions,}, {"role": "user", "content": msg}],
    response_format=Response
  )"""
  
  response = client.chat.completions.create(
        model="meta-llama/llama-4-maverick-17b-128e-instruct",
        messages=[{"role": "system", "content": instructions,}, {"role": "user", "content": msg}],
    )
  
  return response.choices[0].message.content.lower().strip()

# Optionally add the 'traceable' decorator to trace the inputs/outputs of this function.
@traceable
def dummy_app(inputs: dict) -> dict:
  return {"answer": "hmm i'm not sure", "reasoning": "i didn't understand the question"}

ls_client = Client()

dataset = ls_client.create_dataset("big questions")

examples = [
  {"inputs": {"question": "how will the universe end"}},
  {"inputs": {"question": "are we alone"}},
]
ls_client.create_examples(dataset_id=dataset.id, examples=examples)

results = evaluate(
  dummy_app,
  data=dataset,
  evaluators=[valid_reasoning]
)

View the evaluation results for experiment: 'left-charge-47' at:
https://smith.langchain.com/o/d9ab5018-45a8-5efd-961d-320c87839c87/datasets/2000609b-7cc5-4e2c-ac78-0370cd35b4fc/compare?selectedSessions=fec2edfd-a602-4e30-98cb-8d0da37141b9




2it [00:21, 10.70s/it]


# Examples
https://docs.smith.langchain.com/evaluation

In [7]:
from langsmith import Client

client = Client()

# Programmatically create a dataset in LangSmith
# For other dataset creation methods, see:
# https://docs.smith.langchain.com/evaluation/how_to_guides/manage_datasets_programmatically
# https://docs.smith.langchain.com/evaluation/how_to_guides/manage_datasets_in_application
dataset = client.create_dataset(
    dataset_name="Sample dataset", description="A sample dataset in LangSmith."
)

# Create examples
examples = [
    {
        "inputs": {"question": "Which country is Mount Kilimanjaro located in?"},
        "outputs": {"answer": "Mount Kilimanjaro is located in Tanzania."},
    },
    {
        "inputs": {"question": "What is Earth's lowest point?"},
        "outputs": {"answer": "Earth's lowest point is The Dead Sea."},
    },
]

# Add examples to the dataset
client.create_examples(dataset_id=dataset.id, examples=examples)

{'example_ids': ['b729b974-6498-4361-8310-f030b6df346d',
  '1b6d09e4-9dda-4a5a-a16e-86a092bcb0eb'],
 'count': 2}

In [8]:
from langsmith import wrappers
from groq import Groq

api_key = os.environ["GROQ_API_KEY"]
client_model = Groq(
    api_key=api_key)
      
# Define the application logic you want to evaluate inside a target function
# The SDK will automatically send the inputs from the dataset to your target function
def target(inputs: dict) -> dict:
    response = client_model.chat.completions.create(
        model="meta-llama/llama-4-maverick-17b-128e-instruct",
        messages=[
            {"role": "system", "content": "Answer the following question accurately"},
            {"role": "user", "content": inputs["question"]},
        ],
    )
    return { "answer": response.choices[0].message.content.strip() }

In [9]:
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT

def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict):
    evaluator = create_llm_as_judge(
        prompt=CORRECTNESS_PROMPT,
        #model="openai:o3-mini",
        model="groq:meta-llama/llama-4-maverick-17b-128e-instruct",
        feedback_key="correctness",
    )
    eval_result = evaluator(
        inputs=inputs,
        outputs=outputs,
        reference_outputs=reference_outputs
    )
    return eval_result

In [10]:
from langsmith import Client, evaluate
experiment_results = evaluate(
    target,
    data="Sample dataset",
    evaluators=[
        correctness_evaluator,
        # can add multiple evaluators here
    ],
    experiment_prefix="first-eval-in-langsmith",
    max_concurrency=2,
)

View the evaluation results for experiment: 'first-eval-in-langsmith-8237023f' at:
https://smith.langchain.com/o/d9ab5018-45a8-5efd-961d-320c87839c87/datasets/aea62e89-af3d-4bb0-98e6-275fd643bfe5/compare?selectedSessions=938e0faa-1b1e-466a-8a03-125a59fd932c




2it [00:02,  1.03s/it]


In [11]:
results

,inputs.question,outputs.answer,outputs.reasoning,error,feedback.valid_reasoning,execution_time,example_id,id
0,are we alone,hmm i'm not sure,i didn't understand the question,None,to determine if the reasoning for the answer i...,0.0,9196a30d-02fb-42ae-bf7c-e38b5e705025,e305427b-2e87-454a-935a-cc04660ec628
1,how will the universe end,hmm i'm not sure,i didn't understand the question,None,"the reasoning ""i didn't understand the questio...",0.0,9c74fedf-899a-4e54-b934-08b355165db9,79212e59-fd24-4b74-aaa8-853dbe3a2666


# Evaluate a chatbot

https://docs.smith.langchain.com/evaluation/tutorials/evaluation

In [13]:
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "QA Example Dataset"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['da8aff26-8bcf-4af3-8ff4-ae6a109bcba0',
  'eecee606-0839-41d6-aaee-fc05ad0edd15',
  'e08734ae-8a9b-4253-b58b-0220f2b9aef0',
  '485fc257-a1c0-4ed7-97f0-ee36fd40ede3',
  'a5c3cc33-99a0-44da-9dee-b17992dffbde'],
 'count': 5}

In [14]:
from groq import Groq

api_key = os.environ["GROQ_API_KEY"]

groq_model = Groq(
    api_key=api_key
)

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following question:
{inputs['question']}
Here is the real answer:
{reference_outputs['answer']}
You are grading the following predicted answer:
{outputs['response']}
Respond with CORRECT or INCORRECT:
Grade:
"""
    response = groq_model.chat.completions.create(
        #model="gpt-4o-mini",
        model="meta-llama/llama-4-maverick-17b-128e-instruct",
        temperature=0,
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content},
        ],
    ).choices[0].message.content.strip()
    return response == "CORRECT"

In [15]:
def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

In [23]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."

def my_app(question: str, model: str = "meta-llama/llama-4-maverick-17b-128e-instruct", instructions: str = default_instructions) -> str:
    response =  groq_model.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content
    
    if isinstance(response, str) and "\n</think>\n\n" in response:
        response = response.split("\n</think>\n\n")[1]
    
    return response
    

In [25]:
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [26]:
from langsmith import evaluate
experiment_results = evaluate(
    ls_target, # Your AI system
    data=dataset_name, # The data to predict and grade over
    evaluators=[concision, correctness], # The evaluators to score the results
    experiment_prefix="meta-llama/llama-4-maverick-17b-128e-instruct", # A prefix for your experiment names to easily identify them
)

View the evaluation results for experiment: 'meta-llama/llama-4-maverick-17b-128e-instruct-119a5b83' at:
https://smith.langchain.com/o/d9ab5018-45a8-5efd-961d-320c87839c87/datasets/ba1609ff-70a6-4a64-b849-1892bbd24274/compare?selectedSessions=c07b4634-2596-41e6-b6f5-1eff9c2676d1




5it [00:06,  1.36s/it]


In [27]:
def ls_target_v2(inputs: str) -> dict:
    return {"response": my_app(inputs["question"], model="deepseek-r1-distill-llama-70b")}

experiment_results = evaluate(
    ls_target_v2,
    data=dataset_name,
    evaluators=[concision, correctness],
    experiment_prefix="deepseek-r1-distill-llama-70b",
)

View the evaluation results for experiment: 'deepseek-r1-distill-llama-70b-ab6b010e' at:
https://smith.langchain.com/o/d9ab5018-45a8-5efd-961d-320c87839c87/datasets/ba1609ff-70a6-4a64-b849-1892bbd24274/compare?selectedSessions=e15e7b5c-8505-4430-ae7b-7b401f5f387f




5it [00:12,  2.47s/it]


In [28]:

instructions_v3 = "Respond to the users question in a short, concise manner (one short sentence). Do NOT use more than ten words."

def ls_target_v3(inputs: str) -> dict:
    response = my_app(
        inputs["question"], 
        model="qwen-qwq-32b",
        instructions=instructions_v3
    )
    return {"response": response}


experiment_results = evaluate(
    ls_target_v3,
    data=dataset_name,
    evaluators=[concision, correctness],
    experiment_prefix="qwen-qwq-32b",
)

View the evaluation results for experiment: 'qwen-qwq-32b-122b69a8' at:
https://smith.langchain.com/o/d9ab5018-45a8-5efd-961d-320c87839c87/datasets/ba1609ff-70a6-4a64-b849-1892bbd24274/compare?selectedSessions=dc458955-8a21-409e-ae97-d87955660273




5it [00:11,  2.39s/it]


**Configurar testes automatizados para execução em CI/**

Agora que executamos isso de forma única, podemos configurá-lo para ser executado de forma automatizada. Podemos fazer isso facilmente, incluindo-o como um arquivo pytest que executamos no CI/CD. Como parte disso, podemos simplesmente registrar os resultados OU definir alguns critérios para determinar se o teste passa ou não. Por exemplo, se eu quisesse garantir que sempre pelo menos 80% das respostas geradas passassem na lengthverificação, poderíamos configurar isso com um teste como:

In [31]:
def test_length_score() -> None:
    """Test that the length score is at least 80%."""
    experiment_results = evaluate(
        ls_target, # Your AI system
        data=dataset_name, # The data to predict and grade over
        evaluators=[concision, correctness], # The evaluators to score the results
    )
    # This will be cleaned up in the next release:
    feedback = client.list_feedback(
        run_ids=[r.id for r in client.list_runs(project_name=experiment_results.experiment_name)],
        feedback_key="concision"
    )
    scores = [f.score for f in feedback]
    print(scores)
    assert sum(scores) / len(scores) >= 0.8, "Aggregate score should be at least .8"

In [32]:
test_length_score()

View the evaluation results for experiment: 'new-ship-19' at:
https://smith.langchain.com/o/d9ab5018-45a8-5efd-961d-320c87839c87/datasets/ba1609ff-70a6-4a64-b849-1892bbd24274/compare?selectedSessions=98bb17cb-67e3-41a5-a577-2b44aca90fbe




5it [00:05,  1.01s/it]


[0.0, 0.0, 1.0, 0.0]


AssertionError: Aggregate score should be at least .8